# Ajja, Naanu, Naale – Reel 1 ("Phone") on a free Colab GPU

**How to run in VS Code:** Select Kernel → **Colab** → **New Colab Server** → choose **GPU (T4)**. Then run the cells top to bottom.

What it does:
1. Makes one realistic starting image per character + the radio (Stable Diffusion XL)
2. Turns each shot into a short vertical clip (Wan 2.2 TI2V-5B, image-to-video)
3. Saves everything to Google Drive (or `/content/reel1` if Drive isn't available)

Start with `TEST_MODE = True` (only shot_02, short and fast). When it looks good, set it to `False` and run from cell 3 again.
If the runtime disconnects, reconnect and run from the top — finished images and clips are skipped.

In [ ]:
# 1. Check the GPU
import subprocess
print(subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"]).decode())

In [ ]:
# 2. Install libraries (a few minutes)
!pip install -q -U diffusers transformers accelerate ftfy sentencepiece imageio imageio-ffmpeg

In [ ]:
# 3. Settings + shot list
TEST_MODE = True          # True = only shot_02, fewer frames/steps. Set False for the full reel.

WIDTH, HEIGHT = 480, 832  # vertical 9:16-ish (multiples of 32). 704x1280 only on bigger GPUs.
FPS = 24
NUM_FRAMES = 49 if TEST_MODE else 121   # 49 ≈ 2s, 121 ≈ 5s (must be 4k+1)
STEPS = 25 if TEST_MODE else 40
GUIDANCE = 5.0
SEED = 42

NEGATIVE = "blurry, low quality, distorted face, deformed hands, extra fingers, cartoon, anime, text, watermark, subtitles, static image, jittery"

# One fixed starting image per character, already in their own setting,
# so every shot of that character starts from the same face.
CHARACTERS = {
    "radio":  "photorealistic close-up of an old wooden valve radio on a wooden table, glowing orange dial, dark room, cinematic lighting, shallow depth of field",
    "ajja":   "photorealistic, 75-year-old South Indian village man, white panche and cotton shawl, round spectacles, grey stubble, warm wrinkled face, sitting on the verandah of an old tiled-roof Karnataka house next to an old wooden radio, warm evening light, cinematic, 35mm film look",
    "bunty":  "photorealistic, 9-year-old South Indian boy in a school uniform, sitting on a sofa in a modern Bengaluru apartment, holding a smartphone, curious big eyes, natural daylight, cinematic",
    "chukki": "photorealistic, 10-year-old South Indian girl, sleek silver jacket, glowing blue hologram wristband, standing in a clean futuristic room with green plants and soft neon light, confident smile, cinematic",
}

SHOTS = [
    {"id": "shot_01", "char": "radio",  "prompt": "The old radio's dial glows brighter, static sparks flicker, light pulses in blue, gold and white, slow camera push-in"},
    {"id": "shot_02", "char": "ajja",   "prompt": "The old man leans towards the radio, raises one eyebrow in puzzlement and talks, slow camera push-in"},
    {"id": "shot_03", "char": "bunty",  "prompt": "The boy proudly holds up his smartphone towards the camera, excited, talking"},
    {"id": "shot_04", "char": "ajja",   "prompt": "The old man shakes his head and smiles with nostalgia, picks up an old paper envelope, talking"},
    {"id": "shot_05", "char": "bunty",  "prompt": "The boy's jaw drops, eyes go wide in shock, he slowly lowers his phone, camera slowly zooms in on his face"},
    {"id": "shot_06", "char": "chukki", "prompt": "The girl taps her temple with one finger, her wristband glows bright blue, she smiles smugly and talks"},
    {"id": "shot_07", "char": "ajja",   "prompt": "The old man freezes, eyes go wide, glances nervously over his shoulder towards the house, then leans in and whispers to the radio"},
    {"id": "shot_08", "char": "bunty",  "prompt": "The boy bursts out laughing, holding his stomach, leaning back on the sofa"},
]
if TEST_MODE:
    SHOTS = [s for s in SHOTS if s["id"] == "shot_02"]
print("Shots to make:", [s["id"] for s in SHOTS])

In [ ]:
# 4. Output folder (Google Drive if available)
import os
OUT = "/content/reel1"
try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/ajja_naanu_naale/reel1"
    print("Saving to Google Drive")
except Exception as e:
    print("Drive not available, saving on the Colab server only:", e)
IMG_DIR, CLIP_DIR = f"{OUT}/characters", f"{OUT}/clips"
os.makedirs(IMG_DIR, exist_ok=True); os.makedirs(CLIP_DIR, exist_ok=True)
print(OUT)

In [ ]:
# 5. Character images with SDXL (skips ones that already exist)
#    Check them! If one looks bad, delete that file, change SEED, and run this cell again.
import torch, gc
from diffusers import StableDiffusionXLPipeline, AutoencoderKL
from IPython.display import display
from PIL import Image

needed = sorted({s["char"] for s in SHOTS})
missing = [c for c in needed if not os.path.exists(f"{IMG_DIR}/{c}.png")]
if missing:
    vae = AutoencoderKL.from_pretrained("madebyollin/sdxl-vae-fp16-fix", torch_dtype=torch.float16)
    sdxl = StableDiffusionXLPipeline.from_pretrained(
        "stabilityai/stable-diffusion-xl-base-1.0", vae=vae, torch_dtype=torch.float16, variant="fp16"
    ).to("cuda")
    for c in missing:
        g = torch.Generator("cuda").manual_seed(SEED)
        img = sdxl(prompt=CHARACTERS[c] + ", highly detailed, realistic skin, sharp focus",
                   negative_prompt=NEGATIVE, width=768, height=1344,
                   num_inference_steps=35, guidance_scale=6.0, generator=g).images[0]
        img.save(f"{IMG_DIR}/{c}.png"); print("saved", c)
    del sdxl, vae; gc.collect(); torch.cuda.empty_cache()

for c in needed:
    im = Image.open(f"{IMG_DIR}/{c}.png"); im.thumbnail((256, 448)); print(c); display(im)

In [ ]:
# 6. Encode the text prompts, then free that memory (keeps the free T4 from running out)
import torch, gc
from diffusers import WanImageToVideoPipeline

MODEL = "Wan-AI/Wan2.2-TI2V-5B-Diffusers"
todo = [s for s in SHOTS if not os.path.exists(f"{CLIP_DIR}/{s['id']}.mp4")]
EMBEDS = {}
if todo:
    enc = WanImageToVideoPipeline.from_pretrained(MODEL, transformer=None, vae=None, torch_dtype=torch.bfloat16)
    enc.text_encoder.to("cuda")
    with torch.no_grad():
        for s in todo:
            pe, ne = enc.encode_prompt(prompt=s["prompt"], negative_prompt=NEGATIVE,
                                       do_classifier_free_guidance=True, max_sequence_length=512, device="cuda")
            EMBEDS[s["id"]] = (pe.cpu(), ne.cpu())
    del enc; gc.collect(); torch.cuda.empty_cache()
print("Encoded:", list(EMBEDS))

In [ ]:
# 7. Generate the clips (slow on a free T4 — many minutes per clip)
import torch, gc, time
from diffusers import AutoencoderKLWan, WanImageToVideoPipeline
from diffusers.utils import export_to_video
from PIL import Image

if EMBEDS:
    vae = AutoencoderKLWan.from_pretrained(MODEL, subfolder="vae", torch_dtype=torch.float32)
    pipe = WanImageToVideoPipeline.from_pretrained(MODEL, vae=vae, text_encoder=None, tokenizer=None,
                                                   torch_dtype=torch.bfloat16)
    try:
        pipe.vae.enable_tiling()
    except Exception:
        pass
    pipe.enable_model_cpu_offload()   # loads parts onto the GPU only when needed

    for s in todo:
        t0 = time.time()
        start = Image.open(f"{IMG_DIR}/{s['char']}.png").convert("RGB").resize((WIDTH, HEIGHT))
        pe, ne = EMBEDS[s["id"]]
        frames = pipe(image=start, prompt_embeds=pe.to("cuda"), negative_prompt_embeds=ne.to("cuda"),
                      height=HEIGHT, width=WIDTH, num_frames=NUM_FRAMES,
                      num_inference_steps=STEPS, guidance_scale=GUIDANCE,
                      generator=torch.Generator("cuda").manual_seed(SEED)).frames[0]
        export_to_video(frames, f"{CLIP_DIR}/{s['id']}.mp4", fps=FPS)
        print(f"{s['id']} done in {(time.time()-t0)/60:.1f} min")
        gc.collect(); torch.cuda.empty_cache()
print("All done →", CLIP_DIR)

In [ ]:
# 8. Preview the clips
from IPython.display import Video, display
for f in sorted(os.listdir(CLIP_DIR)):
    print(f); display(Video(f"{CLIP_DIR}/{f}", embed=True, width=240))